# (Kriptográfiai) Hash függvények

Algoritmusok és adatszerkezetek tárgyon: Legyen $U$ a kulcshalmaz, $m$ a tábla mérete. Ekkor $h: U \rightarrow 0..(m-1)$ a hash függvény és a $k$ kulcsú adatot a tábla $T[h(k)]$ elemében tároljuk. Feltesszük, hogy $|U| >> m$.
- előny: $\mathcal{O}(1)$ futási idő
- a $h$ függvény *tömörít*

*Kulcsütközés*: Különböző $k_1,k_2$ kulcsokra $h(k_1) = h(k_2)$. Mivel $|U| >> m$, biztosan lesznek üzközések. Ezt el szeretnénk kerülni, de feloldható pl. láncolással. 

Kriptográfiában ennél többet kérünk: ne lehessen polinom időben ütközést találni.

**Definíció**: $H: \{0,1\}^* \rightarrow \{0,1\}^n$

Néhány gyakran használt hash algoritmus:

| Algoritmus | Hash hossza | Ütközés  | Megjelenés |
|------------|-------------|----------|------------|
| MD5        | 128 bit     | explicit | 1992       |
| SHA-1      | 160 bit     | explicit | 1995       |
| SHA-256    | 256 bit     | -        | 2001       |
| SHA-512    | 512 bit     | -        | 2001       |
| SHA3-256   | 256 bit     | -        | 2016       |
| SHA3-512   | 512 bit     | -        | 2016       |
| Blake2b    | 512 bit     | -        | 2015       |
| Blake2s    | 256 bit     | -        | 2015       |

[Python hashlib](https://docs.python.org/3/library/hashlib.html)

In [3]:
import hashlib

In [4]:
hashlib.algorithms_guaranteed

{'blake2b',
 'blake2s',
 'md5',
 'sha1',
 'sha224',
 'sha256',
 'sha384',
 'sha3_224',
 'sha3_256',
 'sha3_384',
 'sha3_512',
 'sha512',
 'shake_128',
 'shake_256'}

In [5]:
hashlib.algorithms_available

{'blake2b',
 'blake2s',
 'md5',
 'md5-sha1',
 'sha1',
 'sha224',
 'sha256',
 'sha384',
 'sha3_224',
 'sha3_256',
 'sha3_384',
 'sha3_512',
 'sha512',
 'sha512_224',
 'sha512_256',
 'shake_128',
 'shake_256',
 'sm3'}

In [6]:
hashlib.md5(b'hello').hexdigest()

'5d41402abc4b2a76b9719d911017c592'

In [7]:
hashlib.md5(b'hello').digest()

b']A@*\xbcK*v\xb9q\x9d\x91\x10\x17\xc5\x92'

In [8]:
mystr = b'hello'
myhash = hashlib.md5(mystr)
myhash

<md5 _hashlib.HASH object @ 0x7f534d419f70>

In [9]:
myhash.update(b'there')
myhash

<md5 _hashlib.HASH object @ 0x7f534d419f70>

In [10]:
myhash.hexdigest()

'c6f7c372641dd25e0fddf0215375561f'

## Ütközéskeresés a születésnap-paradoxont kihasználva

**Kérdés**: Mi annak a valószínűsége, hogy $q$ számú ember közül két ember születésnapja ugyanarra a napra esik? (Feltesszük, hogy minden nap egyenlő valószínűséggel születik valaki és nincs szökőév)

Ezzel analóg: Legyenek $y_i = H(x_i)$ hash-ek. Mi annak a valószínűsége, hogy $y_i = y_j$, ha $i \ne j$?

**Kérdés**: Praktikus-e egy ilyen támadás?

### Small-space birthday attack

Az előzőhöz hasonló futási idejű, de *konstans* memóriaigényű támadás:

Bemenet: $H : \{0,1\}^* \rightarrow \{0, 1\}^n$

Kimenet: Különböző $x,x'$, amire $H(x) = H(x')$

\begin{array}{l}
    x_0 \leftarrow \{0, 1\}^{n+1} \\
    x' := x := x_0 \\
    \mathbf{for}\;i = 1,2,\dots \mathbf{do}: \\
    \quad x := H(x) \\
    \quad x' := H(H(x')) \\
    \quad \mathbf{if}\;x=x'\quad \mathbf{break} \\
    x' := x, x := x_0 \\
    \mathbf{for}\;j=1\;\text{to}\;i: \\
    \quad \mathbf{if}\;H(x) = H(x')\;\mathbf{return}\;x,x'\;\text{and}\;\mathbf{halt} \\
    \mathbf{else}\;x:=H(x), x':=H(x') \\
\end{array}

## Hash függvények blokk titkosítókból

**Davies-Meyer konstrukció**: Legyen $F$ egy blokk titkosító $n$-bit hosszú kulcsokkal és $\ell$-bit hosszú blokkal. Legyen $h: \{0,1\}^{n + \ell} \rightarrow \{0,1\}^\ell$ úgy, hogy $h_i(m_i, h_{i-1}) = F(m_i, h_{i-1}) \oplus h_{i-1}$

In [11]:
from des import *
from utils import *

In [12]:
def davies_meyer(msg: bytes) -> bytes:
    # msg -> 8 bytes hash
    h0 = bytes.fromhex(f'0123456789abcdef')
    for b in range(0, len(msg), 8):
        msg_block = msg[b:b+8]
        des = DES_CBC(msg_block, iv=bytes(8)) # iv fixed -> hash is determinated
        ct = des.encrypt(h0, False)[8:]
        h0 = xor_strings(ct, h0)
    return h0

In [13]:
davies_meyer(b'cryptography0123').hex()

'd55729f9bc8f61e3'

In [14]:
davies_meyer(bytes.fromhex('0101010101010101')), davies_meyer(bytes(8)) # same hash

(b'`X\x7fka[\xbc\xef', b'`X\x7fka[\xbc\xef')

**Feladat**: Legyen $h: \{0,1\}^{128} \rightarrow \{0,1\}^{64}$ úgy, hogy $h(m_1, m_2) = \text{DES}(m_1, \text{DES}(m_2, 0^{64}))$ és $|m_1| = |m_2| = 64$. Adjunk explicit ütközést $h$-ra!

In [15]:
def h(m1, m2):
    des1 = DES_CBC(m1, iv=bytes(8))
    des2 = DES_CBC(m2, iv=bytes(8))
    
    a = des2.encrypt(bytes(8), False)[8:]
    b = des1.encrypt(a, False)[8:]
    return b

In [16]:
h(bytes.fromhex('0101010101010101'), bytes(8)) 

b'\x00\x00\x00\x00\x00\x00\x00\x00'

In [17]:
h(bytes.fromhex('fefefefefefefefe'), bytes.fromhex('ffffffffffffffff'))

b'\x00\x00\x00\x00\x00\x00\x00\x00'

## Jelszavak törése

**Feladat**: Adottak az alábbi fájlok, amik felhasználóneveket és jelszavakat tartalmaznak. A jelszavak valamilyen (ismeretlen) algoritmussal hash-elve vannak. Próbáljuk meg feltörni a jelszavakat! 
- `hashed_pwd_1.txt`
- `hashed_pwd_2.txt`

Adott egy `pwd.txt` fájl, ami 200 jelszót tartalmaz. Ezt felhasználhatjuk a töréshez.

[200 leggyakoribb jelszó](https://nordpass.com/most-common-passwords-list/)

In [18]:
!head hashed_pwd_1.txt

misty65 8d969eef6ecad3c29a3a629280e686cf0c3f5d5a86aff3ca12020c923adc6c92
garymartinez 5994471abb01112afcc18159f6cc74b4f511b99806da59b3caf5a9c173cacfc5
carradam 15e2b0d3c33891ebb0f1ef609ec419420c20e320ce94c65fbc8c3312448eb225
millsamy 65e84be33532fb784c48129675f9eff3a682b27168c0ea744b2cf58ee02337c5
charlesmorales ef797c8118f02dfb649607dd5d3f8c7623048c9c063d532cc95c5ed7a898a64f
walter71 248b646537648c1fbdeb42b56771dbdb42129e8bab527ff551a1f49ce499464f
stephen65 5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8
mauricebennett 8bb0cf6eb9b17d0f7d22b456f121257dc1254e1f01665370476383ea776df414
michaelcameron 7aa7ecc2a319385e1fd583eeaa85e8e32b7c90345991926762f0d40eeb8b567f
lauragarrison 03ac674216f3e15c761ee1a5e255f067953623c8b388b4459e13f978d7c846f4


In [19]:
!head hashed_pwd_2.txt

head: cannot open 'hashed_pwd_2.txt' for reading: No such file or directory


### Dictionary attack

Cél: futási (törési) idő csökkentése egy előre kiszámolt adathalmaz segítségével (ezt potenciálisan sokáig tarthat kiszámolni)

[1,493,677,782 jelszó](https://crackstation.net/crackstation-wordlist-password-cracking-dictionary.htm), 15 GB

In [20]:
pwd_dict = {}
with open('pwd.txt') as infile:
    for pwd in infile:
        pwd = pwd.strip()
        h = hashlib.sha256(pwd.encode()).hexdigest()
        pwd_dict[pwd.encode()] = h
pwd_dict

{b'123456': '8d969eef6ecad3c29a3a629280e686cf0c3f5d5a86aff3ca12020c923adc6c92',
 b'12345': '5994471abb01112afcc18159f6cc74b4f511b99806da59b3caf5a9c173cacfc5',
 b'123456789': '15e2b0d3c33891ebb0f1ef609ec419420c20e320ce94c65fbc8c3312448eb225',
 b'qwerty': '65e84be33532fb784c48129675f9eff3a682b27168c0ea744b2cf58ee02337c5',
 b'12345678': 'ef797c8118f02dfb649607dd5d3f8c7623048c9c063d532cc95c5ed7a898a64f',
 b'jelszo': '248b646537648c1fbdeb42b56771dbdb42129e8bab527ff551a1f49ce499464f',
 b'password': '5e884898da28047151d0e56f8dc6292773603d0d6aabbdd62a11ef721d1542d8',
 b'1234567': '8bb0cf6eb9b17d0f7d22b456f121257dc1254e1f01665370476383ea776df414',
 b'attila': '7aa7ecc2a319385e1fd583eeaa85e8e32b7c90345991926762f0d40eeb8b567f',
 b'1234': '03ac674216f3e15c761ee1a5e255f067953623c8b388b4459e13f978d7c846f4',
 b'asdasd': '5fd924625f6ab16a19cc9807c7c506ae1813490e4ba675f843d5a10e0baacdb8',
 b'szerelem': '881e5d26bad72d500febf35921004707eea2cf4c68beac6b6ad0f6a538afc3b3',
 b'samsung': '968e2d5b08687bf4299

In [21]:
pwd_dict = {v: k for k, v in pwd_dict.items()}
with open('hashed_pwd_1.txt') as infile:
    for line in infile:
        uname, hashed_pwd = line.strip().split()
        
        print(f'{uname} --> {pwd_dict[hashed_pwd]}')

misty65 --> b'123456'
garymartinez --> b'12345'
carradam --> b'123456789'
millsamy --> b'qwerty'
charlesmorales --> b'12345678'
walter71 --> b'jelszo'
stephen65 --> b'password'
mauricebennett --> b'1234567'
michaelcameron --> b'attila'
lauragarrison --> b'1234'
williamsjaime --> b'asdasd'
bjohnson --> b'szerelem'
christophersalazar --> b'samsung'
etaylor --> b'mmklub'
isnyder --> b'qwertz'
xwatkins --> b'666666'
lonnietaylor --> b'lacika'
woodcarlos --> b'tomika'
matthewturner --> b'macika'
ymccall --> b'zolika'
perezjason --> b'asdfgh'
jonathan21 --> b'123123'
cschmidt --> b'asd123'
cwallace --> b'tigris'
kochrachel --> b'malacka'
bobbyharris --> b'eszter'
tjohnston --> b'petike'
saunderssherry --> b'danika'
friedmananthony --> b'abc123'
stephaniecuevas --> b'killer'
jonathan65 --> b'csillag'
qtorres --> b'dragon'
sandra83 --> b'budapest'
james70 --> b'nemtudom'
langphilip --> b'barcelona'
abbottdustin --> b'654321'
shelby19 --> b'kicsim'
rosaleschristina --> b'gabika'
torresvictor --

### Hash chain, Rainbow table

[Hellman](https://ee.stanford.edu/~hellman/publications/36.pdf) (1980) és [Oechslin](https://lasec.epfl.ch/pub/lasec/doc/Oech03.pdf) (2003)

Adott egy $H: \{0, 1\}^* \rightarrow \{0,1\}^n$ hash függvény és a jelszavak egy véges $P$ halmaza. 

Cél: Készítsünk egy olyan adathalmazt, amiben egy adott $h$ hash esetén 
- vagy találunk olyan $p \in P$ jelszót, amire $h = H(p)$, vagy
- nincs olyan $p \in P$, amire $h = H(p)$ teljesül.

Ötlet: definiáljunk egy *redukáló* $R: \{0,1\}^n \rightarrow \{0,1\}^*$ függvényt úgy, hogy $R(h) = p$ (azaz ami egy hashből készít egy $P$-beli értéket). **Vigyázat: Ez nem a hash inverze!!!**

Például:

$\text{abcdef} \rightarrow_\text{H} \text{18cab0cc42} \rightarrow_\text{R} \text{bla4b1} \rightarrow_\text{H} \text{a7f2bba089} \rightarrow_\text{R} \text{keb5ca}$

Egy ilyen sorozatnak az első és utolsó elemét tároljuk csak (kezdő- és végpont). Hogyan keressük meg a jelszót egy adott $h$ hash értékre?
1. Alkalmazzuk az $R$ és $H$ függvényeket egymás után addig, amíg az $R$ által visszaadott érték nem egyezik meg egy végponttal.
2. Az adott végponthoz tartozó kezdőponttól kezdve számoljuk ki addig a láncot, amíg meg nem kapjuk azt a jelszót, ami a hash-t eredményezte.
    - Ez nem feltétlenül egyértelmű: másik jelszó is eredményeztheti ugyanazt a hash-t.

In [22]:
# Az osszes lehetseges jelszo: angol kisbetus abc elso 5 karakterebol allo 5 hosszu jelszavak (5^5 lehetseges jelszo)
# A veges reszhalmaz: Random 1000 jelszo a lehetsegesek kozul
# Hash fuggveny: MD5
# Redukalo fuggeny: MD5 hashbol az elso 5 karakter

In [34]:
import hashlib
from string import ascii_lowercase
import random
from itertools import product

In [27]:
def reduction(msg: str) -> bytes:
    return ''.join(filter(lambda x: x.isalpha(), msg))[:5].encode()

In [28]:
hashlib.md5(b'hello').hexdigest()

'5d41402abc4b2a76b9719d911017c592'

In [29]:
reduction('5d41402abc4b2a76b9719d911017c592')

b'dabcb'

In [42]:
def create_hashchain(h, red, k, initial_pwds):
    hashchain = {}
    for pwd in initial_pwds:
        temp = pwd.encode()
        for i in range(k):
            temp_hash = h(temp).hexdigest()
            temp = red(temp_hash)
            
        hashchain[pwd] = temp
    return hashchain

In [43]:
all_pwds = list(product('abcde', repeat=5))
initial_pwds = [''.join(i) for i in random.sample(all_pwds, k=1000)]

In [44]:
hash_chain = create_hashchain(hashlib.md5, reduction, 100, initial_pwds)
hash_chain

{'bbaab': b'ecdaf',
 'ebbba': b'bacdd',
 'eddea': b'eacda',
 'ebbbd': b'bfbfd',
 'ddcee': b'eacda',
 'caeac': b'aaeeb',
 'ddceb': b'eeecb',
 'eabce': b'cfbca',
 'dbded': b'eeeff',
 'aecec': b'feaba',
 'bacea': b'bbfac',
 'ddccc': b'effdb',
 'acced': b'cdeee',
 'aaacb': b'fadcf',
 'eccec': b'eacda',
 'abbea': b'bbfac',
 'dacdb': b'cdcbc',
 'dbdac': b'fcfce',
 'cddee': b'cfdfe',
 'ddaec': b'fbfff',
 'baeda': b'ecdaf',
 'eaaeb': b'cdbac',
 'ebbcb': b'dcfaf',
 'bacdb': b'dabaf',
 'ccebe': b'cbfbb',
 'cdcde': b'fcbbf',
 'dcbeb': b'bbbac',
 'ebcbc': b'cfdac',
 'ecbdd': b'cfbca',
 'beced': b'effdb',
 'edace': b'dcfaf',
 'caeab': b'feaba',
 'abbae': b'dffbd',
 'eedbc': b'bcdfb',
 'cdced': b'fbfff',
 'adadd': b'ceebd',
 'adcac': b'ebfee',
 'ccdbd': b'dcefa',
 'aeebb': b'dffdd',
 'cbcbe': b'ecdaf',
 'aeddd': b'bbbac',
 'cdecd': b'fbfff',
 'edbae': b'fbfff',
 'eddbb': b'bbcda',
 'aabde': b'feaba',
 'abbdb': b'cfdac',
 'ababb': b'aaeeb',
 'eedec': b'dcafc',
 'babbd': b'acfeb',
 'cceab': b'fdabe',


In [49]:
hashlib.md5(b'bbdca').hexdigest()

'274afccaa83a3a703c8a9836b8dc93a8'

In [53]:
endpoint = reduction('5d41402abc4b2a76b9719d911017c592')
endpoint in hash_chain.values()

False

In [54]:
k = 0
while endpoint not in hash_chain.values() or k <= 20:
    e = hashlib.md5(endpoint).hexdigest()
    endpoint = reduction(e)
    k += 1
endpoint in hash_chain.values()

True

Problémák: 
1. A láncon belül is lehetnek ütközések, azaz egy lánc beleolvad egy másik láncba valahol. Ezt nagyon nehéz detektálni.
2. A redukáló függvényt jól kell megválasztani

Az ütközés problémáját javítja a *Rainbow table*: annyi redukálófüggvényt választunk, amilyen hosszú láncot szeretnénk

## Salt, Pepper

1. Salt: random sztring, hozzáappendeljük a jelszóhoz hash-elés előtt. Adatbázisban tároljuk.
    - $h(p + s)$, $h(s + p)$, $h(h(p) + s)$
2. Pepper: titkos sztring, jellemzően HSM-ben tárolva.

Real World Example: [Hogyan tárolja a Dropbox a jelszavakat (2016)](https://dropbox.tech/security/how-dropbox-securely-stores-your-passwords)

További érdekességek:
- SHA1 algoritmusban explicit ütközés: https://shattered.io/
    - 9,223,372,036,854,775,808 SHA-1 compressions
    - 110 GPU, 1 év számolás
    - Brute force (születésnap) ugyanez 12,000,000 GPU, 1 év számolás
    - Deprecated in 2011 by NIST
    - Firefox 2017. február 27-én vonta ki
- MD5 ütközés keresésre
    - [Egy támadás](https://iacr.org/archive/eurocrypt2005/34940019/34940019.pdf)
    - legjobb támadás $2^{16}$ hash számolásából talál ütközést (telefonon 30 másodperc)
- SHA-1, SHA-224, SHA-256, SHA-384, SHA-512, SHA-512/224, SHA-512/256 [standard](https://csrc.nist.gov/publications/detail/fips/180/4/final)
- SHA3-224, SHA3-256, SHA3-384, SHA3-512 [standard](https://csrc.nist.gov/publications/detail/fips/202/final)
    - És az ezekből levezetett [struktúrák](https://csrc.nist.gov/publications/detail/sp/800-185/final)